# 23 — Q-Uniform Binning

PDF and Rietveld want intensities on a **Q-uniform** grid (constant ΔQ),
but the detector bins are naturally R-uniform (constant ΔR). The mapping
`Q = 4π/λ · sin(θ)` is non-linear, so a uniform-R cake produces a
non-uniform-Q profile.

`build_q_bin_edges_in_R` returns the R-bin edges corresponding to a
uniformly-spaced Q grid given (Lsd, λ, px). Use as the `RBinEdges`
argument to any binner.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2.corrections import build_q_bin_edges_in_R

R_edges_for_uniform_Q = build_q_bin_edges_in_R(
    Q_min_invA=0.5, Q_max_invA=20.0, n_Q_bins=2000,
    Lsd_um=940000.0, wavelength_A=0.184139, pxY_um=200.0,
)
print(f'returned {len(R_edges_for_uniform_Q)} R edges spanning '
      f'{float(R_edges_for_uniform_Q.min()):.1f} -- {float(R_edges_for_uniform_Q.max()):.1f} px')
print(f'(corresponds to Q = 0.5 -- 20 inv-A, uniformly spaced)')

## Use it as a custom bin grid

Most binners (`HardBinGeometry`, `SoftBinGeometry`, etc.) accept either
a `(RMin, RMax, RBinSize)` triple or an explicit `RBinEdges` tensor. Pass
the Q-uniform edges as the latter:

```python
geom = HardBinGeometry.from_spec(spec, RBinEdges=R_edges_for_uniform_Q)
out  = integrate_hard(img, geom)
# out.profile is now on a Q-uniform grid implicitly.
```

The output 1-D profile is then linear in Q — the right input format for
TOPAS, PDFgui, GSAS-II, etc.